In [3]:
# Célula 1: Instalação das dependências
%pip install pandas "sqlalchemy>=2.0" python-dotenv mysql-connector-python oracledb boto3

   ---------------------------------------- 0.0/13.9 MB ? eta -:--:--
   ---- ----------------------------------- 1.6/13.9 MB 8.4 MB/s eta 0:00:02
   ----------- ---------------------------- 3.9/13.9 MB 10.2 MB/s eta 0:00:01
   ------------------ --------------------- 6.6/13.9 MB 10.8 MB/s eta 0:00:01
   -------------------------- ------------- 9.2/13.9 MB 11.0 MB/s eta 0:00:01
   --------------------------------- ------ 11.5/13.9 MB 11.2 MB/s eta 0:00:01
   ---------------------------------------- 13.9/13.9 MB 11.1 MB/s eta 0:00:00

   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------- ----------------------- 2/5 [botocore]
   ---------------- ----------------------- 2/5 [botocore]
   ---------------- ----------------------- 2/5 [botocore]
   ---------------- ----------------------- 2/5 [botocore]
   ---------------- ----------------------- 2/5 [botocore]
   ---------------- ----------------------- 2/5 [


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Importações necessárias
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
import os
from dotenv import load_dotenv
import oracledb
from datetime import datetime
import boto3
import json
import pandas as pd


# Carrega as variáveis do arquivo .env
load_dotenv()

# --- Carregar Variáveis de Ambiente ---
# Oracle
oracle_user = os.getenv('ORACLE_USER_V2')
oracle_password = os.getenv('ORACLE_PASSWORD_V2')
oracle_dsn = os.getenv('ORACLE_DSN_V2')
oracle_instant_client_path = os.getenv('ORACLE_INSTANT_CLIENT_PATH')

# MariaDB
mariadb_user = os.getenv('MARIADB_USER')
mariadb_password_raw = os.getenv('MARIADB_PASSWORD')
mariadb_host = os.getenv('MARIADB_HOST')
mariadb_port = os.getenv('MARIADB_PORT')
mariadb_database = os.getenv('MARIADB_DATABASE')

print("Variáveis de ambiente carregadas com sucesso!")
# Inicializa o Oracle Client (necessário no Windows e em alguns setups do Linux)
try:
    oracledb.init_oracle_client(lib_dir=oracle_instant_client_path)
    print("Oracle Instant Client inicializado com sucesso.")
except Exception as e:
    print(f"Erro ao inicializar o Oracle Instant Client: {e}")
    print("Verifique se o caminho em ORACLE_INSTANT_CLIENT_PATH está correto no seu .env")

srvhmddb003-producao-scan:1521/pdbprd.hmaterdei.com.br
Variáveis de ambiente carregadas com sucesso!
Erro ao inicializar o Oracle Instant Client: DPI-1047: Cannot locate a 64-bit Oracle Client library: "The specified module could not be found". See https://python-oracledb.readthedocs.io/en/latest/user_guide/initialization.html for help
Help: https://python-oracledb.readthedocs.io/en/latest/user_guide/troubleshooting.html#dpi-1047
Verifique se o caminho em ORACLE_INSTANT_CLIENT_PATH está correto no seu .env


In [5]:
# Célula 3: Consulta ao MariaDB
mariadb_database = "surgery_flow_prod" 
# 1. Defina o nome do médico que você quer consultar
#nome_do_medico = 'RODRIGO FABIANO GUEDES LEITE' 

# 2. Crie a string de conexão para o MariaDB
mariadb_password_encoded = quote_plus(mariadb_password_raw)
print("Senha preparada para a conexão.")

# 3. Crie o motor de conexão com a senha codificada e o banco de dados especificado
try:
    # A URL de conexão agora está correta e completa
    connection_url = (
        f"mysql+mysqlconnector://{mariadb_user}:{mariadb_password_encoded}"
        f"@{mariadb_host}:{mariadb_port}/{mariadb_database}"
    )
    mariadb_engine = create_engine(connection_url)
    print("Motor de conexão com MariaDB criado com sucesso!")

except Exception as e:
    print(f"Erro ao criar motor de conexão MariaDB: {e}")
    # Encerra a execução da célula se a conexão falhar
    raise


# 3. SQL Query para o MariaDB (usando a variável nome_do_medico)
# Adicionei um apelido 'db' na cláusula WITH para consistência.
query_mariadb = f"""
WITH db AS (
    SELECT 
        so.surgical_order_id,
        so.hospital_id,
        h.friendly_name,
        IFNULL(IFNULL(u.name, so.doctor_name), 'SEM REGISTRO') AS doctor_name,
        d.specialty,
        so.patient_name,
        CASE 
            WHEN so.opme = '{{\"solicitations\":[],\"providers\":[]}}' THEN 'Sem OPME' 
            WHEN so.opme IS NULL THEN 'Sem OPME' 
            ELSE 'Com OPME' 
        END AS with_opme,
        so.opme,
        CASE 
            WHEN CHAR_LENGTH(SUBSTRING_INDEX(SUBSTRING(so.procedure, 34, 100), '"', 1)) = 0 THEN SUBSTRING_INDEX(SUBSTRING(so.procedure, 78, 100), '"', 1) 
            WHEN CHAR_LENGTH(SUBSTRING_INDEX(SUBSTRING(so.procedure, 34, 100), '"', 1)) = 1 THEN SUBSTRING_INDEX(SUBSTRING(so.procedure, 36, 100), '"', 1)
            ELSE SUBSTRING_INDEX(SUBSTRING(so.procedure, 34, 100), '"', 1) 
        END AS procedimento_teste,
        so.created_at,
        hi.health_insurance_code,
        hi.health_insurance_name,
        ss.status
    FROM surgery_flow_prod.surgical_order so     
    JOIN surgery_flow_prod.hospital h ON so.hospital_id = h.hospital_id   
    LEFT JOIN surgery_flow_prod.doctor d ON d.doctor_id = so.doctor_id     
    LEFT JOIN surgery_flow_prod.user u ON d.user_id = u.user_id     
    LEFT JOIN surgery_flow_prod.health_insurance hi ON hi.health_insurance_id = so.health_insurance_id
    JOIN surgery_flow_prod.surgical_status ss ON ss.surgical_order_id = so.surgical_order_id AND ss.is_active = TRUE 
    WHERE so.deleted_at IS NULL
    AND so.created_at >= '2025-01-01'
)
SELECT 
    db.surgical_order_id,
    db.hospital_id,
    db.friendly_name,
    db.doctor_name,
    db.specialty,
    db.patient_name,
    db.opme,
    db.procedimento_teste,
    db.created_at,
    db.health_insurance_code,
    db.health_insurance_name
FROM db
WHERE
    db.with_opme LIKE 'Com OPME'
    AND db.status LIKE 'Realizada'
"""

# 4. Executar a query e carregar em um DataFrame
#print(f"\nBuscando cirurgias para o médico: {nome_do_medico}...")
try:
    with mariadb_engine.connect() as connection:
        df_mariadb = pd.read_sql_query(text(query_mariadb), connection)

    print(f"\nConsulta ao MariaDB concluída. {len(df_mariadb)} registros encontrados.")
    display(df_mariadb.head())

    # Contagem das principais cirurgias (Objetivo 2 da sua query MariaDB)
    print("\nContagem de cirurgias por procedimento:")
    contagem_procedimentos = df_mariadb['procedimento_teste'].value_counts()
    display(contagem_procedimentos)

except Exception as e:
    print(f"Ocorreu um erro ao consultar o MariaDB: {e}")
    df_mariadb = pd.DataFrame() # Cria um dataframe vazio para não quebrar o resto do código

Senha preparada para a conexão.
Motor de conexão com MariaDB criado com sucesso!

Consulta ao MariaDB concluída. 16812 registros encontrados.


,surgical_order_id,hospital_id,friendly_name,doctor_name,specialty,patient_name,opme,procedimento_teste,created_at,health_insurance_code,health_insurance_name
0,791955,6,Contorno,BRUNO FARES DIAS,ORTOPEDIA E TRAUMATOLOGIA,RENATO BRASILEIRO DE LIMA,"{""solicitations"":[{""description"":""HASTE SUPRA ...",FRATURAS DE TÍBIA ASSOCIADA OU NÃO A FÍBULA (I...,2025-01-01 00:27:21,10,AMIL
1,791967,8,Salvador/Bahia,DANIEL FRANCISCO VIRIATO DOS SANTOS,CIRURGIA GERAL,MARIA UMBELINA GONCALVES CONCEICAO,"{""solicitations"":[{""description"":""CATETER DUPL...","IMPLANTE DE CATETER VENOSO CENTRAL POR PUNCAO,...",2025-01-01 12:29:10,28,CASSI
2,791971,8,Salvador/Bahia,DAVI ARAUJO VEIGA ROSARIO,ORTOPEDIA E TRAUMATOLOGIA,LUIZ MARQUES ROCHA FILHO,"{""solicitations"":[{""description"":""HASTE CEFALO...",FRATURAS DO FEMUR - TRATAMENTO CIRURGICO,2025-01-01 13:31:02,414,CNU
3,791974,6,Contorno,CLERISTON LOPES LACERDA,NEUROCIRURGIA,GUILHERME AUGUSTO STIVANIN,"{""solicitations"":[{""description"":""campo adesiv...",HEMATOMA SUBDURAL CRONICO: TRATAMENTO CI,2025-01-01 13:56:49,17,BACEN
4,791975,7,Betim-Contagem,LUIS FERNANDO CARNEIRO VILABOIM,CIRURGIA GERAL,MARIA EDUARDA BRITO SILVA,"{""solicitations"":[{""description"":""TROCARTE DES...",APENDICECTOMIA POR VIDEOLAPAROSCOPIA,2025-01-01 13:59:27,20,BRADESCO



Contagem de cirurgias por procedimento:


procedimento_teste
GASTROPLASTIA PARA OBESIDADE MÓRBIDA POR VIDEO                 722
HISTEROSCOPIA COM RESSECTOSCÓPIO PARA MIOMECTOMIA, POLIPECT    670
COLECISTECTOMIA SEM COLANGIOGRAFIA POR V                       632
URETERORRENOLITOTRIPSIA FLEXIVEL A LASER UNILATERAL            431
REPARO OU SUTURA DE UM MENISCO (JOELHO)                        389
                                                              ... 
REVASCULARIZACAO DISTAL                                          1
ATRIOSSEPTOSTOMIA POR BALAO                                      1
INTRA-OPERATÓRIO                                                 1
MEDULA ÓSSEA, ASPIRAÇÃO PARA MIELOGRAMA                          1
ESOFAGECTOMIA DISTAL SEM TARACOTOMIA                             1
Name: count, Length: 1327, dtype: int64

In [6]:
# Célula 4 (VERSÃO FINAL): Consulta ao OracleDB em Lotes

df_oracle = pd.DataFrame() 

if not df_mariadb.empty:
    # A conexão 'oracle_engine' foi criada em uma célula anterior
    # 3. Criar o motor de conexão usando um "creator"
    # Esta função passa os parâmetros da forma correta para o oracledb
    # Carregar credenciais e o DSN do .env
    oracle_user = os.getenv('ORACLE_USER')
    oracle_password_raw = os.getenv('ORACLE_PASSWORD')
    oracle_dsn_string = os.getenv('ORACLE_DSN') # ex: 'host:porta/service_name'
    
    # 3. Criar o motor de conexão usando um "creator"
    # Esta função passa os parâmetros da forma correta para o oracledb
    def oracle_connection_creator():
        # A CORREÇÃO ESTÁ AQUI: usamos o parâmetro 'dsn'
        return oracledb.connect(
            user=oracle_user, 
            password=oracle_password_raw, 
            dsn=oracle_dsn_string
        )
    
    try:
        oracle_engine = create_engine("oracle+oracledb://", creator=oracle_connection_creator)
        print("Motor de conexão com OracleDB criado com sucesso.")
    except Exception as e:
        print(f"Erro ao criar motor de conexão Oracle: {e}")
        raise
    
    # PASSO 1: Obter a lista completa e única de IDs
    lista_de_ids = df_mariadb['surgical_order_id'].unique().tolist()
    
    # PASSO 2: Quebrar a lista de IDs em lotes de 999 (para ficar abaixo do limite de 1000 do Oracle)
    tamanho_do_lote = 999
    lotes_de_ids = [lista_de_ids[i:i + tamanho_do_lote] for i in range(0, len(lista_de_ids), tamanho_do_lote)]
    
    print(f"A lista de {len(lista_de_ids)} IDs foi dividida em {len(lotes_de_ids)} lotes para consulta.")
    
    lista_de_dataframes_oracle = [] # Lista para guardar os resultados de cada lote

    try:
        # PASSO 3: Iterar sobre cada lote e executar a consulta
        for i, lote in enumerate(lotes_de_ids):
            print(f"Processando lote {i+1} de {len(lotes_de_ids)}...")
            
            ids_formatados = ', '.join(map(str, lote))
            
            query_oracle_lote = f"""
            SELECT
                ac.cd_aviso_cirurgia, 
                itf.cd_pro_fat,
                pf.ds_pro_fat,
                SUM(itf.QT_LANCAMENTO) AS quantidade_utilizada
            FROM
                dbamv.aviso_cirurgia ac
            JOIN
                dbamv.reg_fat rf ON ac.cd_atendimento = rf.cd_atendimento
            JOIN
                dbamv.itreg_fat itf ON rf.cd_reg_fat = itf.cd_reg_fat
            LEFT JOIN
                dbamv.pro_fat pf ON itf.cd_pro_fat = pf.cd_pro_fat
            WHERE
                ac.cd_aviso_cirurgia IN ({ids_formatados})
                AND itf.cd_gru_fat = 5 
            GROUP BY
                ac.cd_aviso_cirurgia, 
                itf.cd_pro_fat,
                pf.ds_pro_fat
            """
            
            with oracle_engine.connect() as connection:
                df_lote = pd.read_sql_query(text(query_oracle_lote), connection)
                lista_de_dataframes_oracle.append(df_lote)
        
        # PASSO 4: Juntar todos os resultados em um único DataFrame
        if lista_de_dataframes_oracle:
            df_oracle = pd.concat(lista_de_dataframes_oracle, ignore_index=True)
            print(f"\nConsulta ao OracleDB concluída. {len(df_oracle)} registros de materiais encontrados em todos os lotes.")
            display(df_oracle.head())
        else:
            print("\nNenhum resultado retornado da consulta ao OracleDB.")

    except Exception as e:
        print(f"Ocorreu um erro ao consultar o OracleDB em lotes: {e}")

else:
    print("Nenhum dado encontrado no MariaDB. A consulta ao OracleDB não será executada.")

display(df_oracle[df_oracle['cd_aviso_cirurgia'] == 804306])

Motor de conexão com OracleDB criado com sucesso.
A lista de 16812 IDs foi dividida em 17 lotes para consulta.
Processando lote 1 de 17...
Processando lote 2 de 17...
Processando lote 3 de 17...
Processando lote 4 de 17...
Processando lote 5 de 17...
Processando lote 6 de 17...
Processando lote 7 de 17...
Processando lote 8 de 17...
Processando lote 9 de 17...
Processando lote 10 de 17...
Processando lote 11 de 17...
Processando lote 12 de 17...
Processando lote 13 de 17...
Processando lote 14 de 17...
Processando lote 15 de 17...
Processando lote 16 de 17...
Processando lote 17 de 17...

Consulta ao OracleDB concluída. 54331 registros de materiais encontrados em todos os lotes.


,cd_aviso_cirurgia,cd_pro_fat,ds_pro_fat,quantidade_utilizada
0,794542,09066332,TELA DE MARLEX EM POLIPROPILENO 30CMX30CM REF ...,1
1,795205,09002799,CARGA AZUL 75MM PARA GRAMPEADOR - REF- TCR75,2
2,792721,09004626,GRAMPEADOR LINEAR CORTANTE 75MM REF- TLC75,2
3,791988,09004644,TRANSDUTOR DE PRESSAO DESCARTAVEL REF- PX260,4
4,793025,09004644,TRANSDUTOR DE PRESSAO DESCARTAVEL REF- PX260,1


,cd_aviso_cirurgia,cd_pro_fat,ds_pro_fat,quantidade_utilizada
11592,804306,09073846,TESOURA ULTRACISION ERGONOMIC HARMONIC 5MM REF...,1
12048,804306,09085139,TROCARTE DESCARTAVEL 12MM SEM AGULHA REF B12LT,1
12177,804306,09126714,ENDOGRAMPEADOR LINEAR ARTICULADO 60MM REF: PSE...,1
12349,804306,09189182,CARGA 60MM VERDE ENDOGRAMPEADOR POWER PLUS REF...,2
12635,804306,09127565,CARGA 60MM AZUL ENDOGRAMPEADOR POWER PLUS REF-...,5


In [7]:
# Célula 4.5 (ATUALIZADA): Consulta ao MariaDB para Dados de Autorização (Fonte MMD)

import json
from pandas import json_normalize

print("Iniciando a busca por dados de autorização de OPME (MMD)...")
df_autorizacao = pd.DataFrame()

if not df_mariadb.empty:
    try:
        lista_de_ids = df_mariadb['surgical_order_id'].unique().tolist()
        ids_formatados = ', '.join(map(str, lista_de_ids))

        # Query agora também busca o JSON original para auditoria
        query_autorizacao = f"""
        SELECT 
            fo.surgical_order_id, 
            fo.opme
        FROM surgery_flow_prod.finalize_surgical_order fo
        WHERE fo.surgical_order_id IN ({ids_formatados})
          AND fo.is_active = TRUE
        """

        with mariadb_engine.connect() as connection:
            df_opme_json = pd.read_sql_query(text(query_autorizacao), connection)
        
        print(f"Consulta de autorização (MMD) concluída. {len(df_opme_json)} registros encontrados.")

        df_opme_json.dropna(subset=['opme'], inplace=True)
        
        registros_processados = []
        for index, row in df_opme_json.iterrows():
            surgical_order_id = row['surgical_order_id']
            opme_json_original = row['opme'] # Guarda o JSON original
            try:
                opme_data = json.loads(opme_json_original)
                
                for item in opme_data:
                    item['surgical_order_id'] = surgical_order_id
                    item['opme_source_json'] = opme_json_original # Adiciona o JSON original a cada item
                    registros_processados.append(item)
            except (json.JSONDecodeError, TypeError):
                continue
        
        df_autorizacao = pd.DataFrame(registros_processados)

        if 'code' in df_autorizacao.columns:
            df_autorizacao.rename(columns={'code': 'cd_pro_fat'}, inplace=True)

        print(f"Processamento do JSON (MMD) concluído. {len(df_autorizacao)} itens extraídos.")
        display(df_autorizacao.head())

    except Exception as e:
        print(f"Ocorreu um erro ao buscar ou processar os dados de autorização (MMD): {e}")
else:
    print("Nenhum dado de cirurgia encontrado. Etapa de autorização (MMD) pulada.")

Iniciando a busca por dados de autorização de OPME (MMD)...
Consulta de autorização (MMD) concluída. 11196 registros encontrados.
Processamento do JSON (MMD) concluído. 38875 itens extraídos.


,cd_pro_fat,description,provider,quantity_solicitation,quantity_authorization,surgical_order_id,opme_source_json
0,None,DIU MIRENA,PACIENTE QUE TRAZ,1,1.0,792231,"[{""code"":null,""description"":""DIU MIRENA "",""pro..."
1,None,1900788171 - KIT CIRURGIA BARIATRICA POR VIDEO...,PANTHER,1,1.0,792175,"[{""code"":null,""description"":""1900788171 - KIT ..."
2,None,SONDA FAUCHER,HOSPITAL,1,1.0,792175,"[{""code"":null,""description"":""1900788171 - KIT ..."
3,None,SENSOR BIS,HOSPITAL,1,1.0,792175,"[{""code"":null,""description"":""1900788171 - KIT ..."
4,None,KIT TROCARTER,MEDCOMERCE,1,1.0,792426,"[{""code"":null,""description"":""KIT TROCARTER"",""p..."


In [8]:
# Célula 4.7 (ATUALIZADA): Consulta ao OracleDB para Dados de Autorização (Fonte MV)

print("Iniciando a busca por dados de autorização de OPME no Oracle (MV)...")
df_autorizacao_oracle = pd.DataFrame()

if not df_mariadb.empty:
    try:
        # Reutilizamos a mesma lógica de lotes da Célula 4
        lista_de_ids = df_mariadb['surgical_order_id'].unique().tolist()
        tamanho_do_lote = 999
        lotes_de_ids = [lista_de_ids[i:i + tamanho_do_lote] for i in range(0, len(lista_de_ids), tamanho_do_lote)]
        
        lista_de_dataframes_aut_oracle = []

        # Itera sobre cada lote e executa a nova consulta
        for i, lote in enumerate(lotes_de_ids):
            print(f"Processando lote {i+1} de {len(lotes_de_ids)} para autorização MV...")
            ids_formatados = ', '.join(map(str, lote))
            
            # ATUALIZAÇÃO AQUI: A query agora busca a qt_solicitada e a qt_autorizada do convênio
            query_aut_oracle_lote = f"""
            SELECT 
               guia.cd_aviso_cirurgia,
               it_guia.cd_pro_fat, 
               it_guia.QT_AUTORIZADO AS mv_qt_solicitada,
               it_guia.QT_AUTORIZADA_CONVENIO AS mv_qt_autorizada
            FROM dbamv.guia 
            JOIN dbamv.it_guia ON guia.cd_guia = it_guia.cd_guia
            WHERE guia.cd_aviso_cirurgia IN ({ids_formatados})
            """
            
            with oracle_engine.connect() as connection:
                df_lote = pd.read_sql_query(text(query_aut_oracle_lote), connection)
                lista_de_dataframes_aut_oracle.append(df_lote)
        
        if lista_de_dataframes_aut_oracle:
            df_autorizacao_oracle = pd.concat(lista_de_dataframes_aut_oracle, ignore_index=True)

        print(f"\nConsulta de autorização no Oracle (MV) concluída. {len(df_autorizacao_oracle)} registros encontrados.")
        display(df_autorizacao_oracle.head())

    except Exception as e:
        print(f"Ocorreu um erro ao consultar autorizações no Oracle: {e}")
else:
    print("Nenhum dado de cirurgia encontrado. Etapa de autorização Oracle (MV) pulada.")

Iniciando a busca por dados de autorização de OPME no Oracle (MV)...
Processando lote 1 de 17 para autorização MV...
Processando lote 2 de 17 para autorização MV...
Processando lote 3 de 17 para autorização MV...
Processando lote 4 de 17 para autorização MV...
Processando lote 5 de 17 para autorização MV...
Processando lote 6 de 17 para autorização MV...
Processando lote 7 de 17 para autorização MV...
Processando lote 8 de 17 para autorização MV...
Processando lote 9 de 17 para autorização MV...
Processando lote 10 de 17 para autorização MV...
Processando lote 11 de 17 para autorização MV...
Processando lote 12 de 17 para autorização MV...
Processando lote 13 de 17 para autorização MV...
Processando lote 14 de 17 para autorização MV...
Processando lote 15 de 17 para autorização MV...
Processando lote 16 de 17 para autorização MV...
Processando lote 17 de 17 para autorização MV...

Consulta de autorização no Oracle (MV) concluída. 167482 registros encontrados.


,cd_aviso_cirurgia,cd_pro_fat,mv_qt_solicitada,mv_qt_autorizada
0,791955,30710057,1.0,1.0
1,791955,00020010,1.0,1.0
2,791955,30727138,1.0,1.0
3,791955,30728126,1.0,1.0
4,791955,09128062,1.0,1.0


In [9]:
# Célula 5 (VERSÃO CORRIGIDA): Corrigindo o nome da coluna

print("Iniciando o merge final com colunas separadas para cada fonte de autorização...")

# PASSO 1: Preparar os dataframes de autorização com colunas específicas
# Prepara o DataFrame do MariaDB (MMD)
df_auth_mmd = df_autorizacao.rename(columns={
    'quantity_solicitation': 'mmd_qt_solicitada',
    # CORREÇÃO AQUI: O nome original da coluna é 'quantity_authorization'
    'quantity_authorization': 'mmd_qt_autorizada',
    'opme_source_json': 'mmd_opme_source_json'
})

# Prepara o DataFrame do Oracle (MV)
df_auth_mv = df_autorizacao_oracle.rename(columns={
    'cd_aviso_cirurgia': 'surgical_order_id'
})

# Agrupamos os dados de autorização para evitar duplicatas no merge.
if not df_auth_mmd.empty:
    df_auth_mmd = df_auth_mmd.groupby(['surgical_order_id', 'cd_pro_fat']).agg(
        mmd_qt_solicitada=('mmd_qt_solicitada', lambda x: x.sum(min_count=1)),
        mmd_qt_autorizada=('mmd_qt_autorizada', lambda x: x.sum(min_count=1)),
        mmd_opme_source_json=('mmd_opme_source_json', 'first')
    ).reset_index()

if not df_auth_mv.empty:
    df_auth_mv = df_auth_mv.groupby(['surgical_order_id', 'cd_pro_fat']).agg(
        mv_qt_solicitada=('mv_qt_solicitada', lambda x: x.sum(min_count=1)),
        mv_qt_autorizada=('mv_qt_autorizada', lambda x: x.sum(min_count=1))
    ).reset_index()


# PASSO 2: Juntar cirurgias com os itens consumidos (base)
df_oracle_conta = df_oracle.rename(columns={'quantidade_utilizada': 'qt_conta'})
df_parcial = pd.merge(
    df_mariadb,
    df_oracle_conta,
    left_on='surgical_order_id',
    right_on='cd_aviso_cirurgia',
    how='left'
)
if 'cd_aviso_cirurgia' in df_parcial.columns:
    df_parcial = df_parcial.drop(columns=['cd_aviso_cirurgia'])
print("Cirurgias e itens de consumo unidos.")

# PASSO 3: Adicionar dados de autorização do MMD
if not df_auth_mmd.empty:
    df_intermediario = pd.merge(
        df_parcial,
        df_auth_mmd,
        on=['surgical_order_id', 'cd_pro_fat'],
        how='left'
    )
    print("Dados de autorização do MMD foram unidos.")
else:
    df_intermediario = df_parcial

# PASSO 4: Adicionar dados de autorização do MV
if not df_auth_mv.empty:
    df_final = pd.merge(
        df_intermediario,
        df_auth_mv,
        on=['surgical_order_id', 'cd_pro_fat'],
        how='left'
    )
    print("Dados de autorização do MV foram unidos.")
else:
    df_final = df_intermediario


# --- Limpeza Final ---
df_final['qt_conta'] = df_final['qt_conta'].fillna(0).astype(int)
df_final['cd_pro_fat'] = df_final['cd_pro_fat'].fillna('N/A')
df_final['ds_pro_fat'] = df_final['ds_pro_fat'].fillna('Nenhum item em conta')


print("\nMerge final concluído!")
print("O DataFrame 'df_final' agora trata corretamente os valores nulos de autorização.")
display(df_final.head())

Iniciando o merge final com colunas separadas para cada fonte de autorização...
Cirurgias e itens de consumo unidos.
Dados de autorização do MMD foram unidos.
Dados de autorização do MV foram unidos.

Merge final concluído!
O DataFrame 'df_final' agora trata corretamente os valores nulos de autorização.


,surgical_order_id,hospital_id,friendly_name,doctor_name,specialty,patient_name,opme,procedimento_teste,created_at,health_insurance_code,health_insurance_name,cd_pro_fat,ds_pro_fat,qt_conta,mmd_qt_solicitada,mmd_qt_autorizada,mmd_opme_source_json,mv_qt_solicitada,mv_qt_autorizada
0,791955,6,Contorno,BRUNO FARES DIAS,ORTOPEDIA E TRAUMATOLOGIA,RENATO BRASILEIRO DE LIMA,"{""solicitations"":[{""description"":""HASTE SUPRA ...",FRATURAS DE TÍBIA ASSOCIADA OU NÃO A FÍBULA (I...,2025-01-01 00:27:21,10,AMIL,09128062,BAINHA-PROTEÇÃO EXTERIOR 12 P/EXPERT TN - REF:...,1,NaN,NaN,NaN,2.0,2.0
1,791955,6,Contorno,BRUNO FARES DIAS,ORTOPEDIA E TRAUMATOLOGIA,RENATO BRASILEIRO DE LIMA,"{""solicitations"":[{""description"":""HASTE SUPRA ...",FRATURAS DE TÍBIA ASSOCIADA OU NÃO A FÍBULA (I...,2025-01-01 00:27:21,10,AMIL,09181974,HASTE EXPERT TN TAMANHIOS DIVERSOS,1,NaN,NaN,NaN,2.0,2.0
2,791955,6,Contorno,BRUNO FARES DIAS,ORTOPEDIA E TRAUMATOLOGIA,RENATO BRASILEIRO DE LIMA,"{""solicitations"":[{""description"":""HASTE SUPRA ...",FRATURAS DE TÍBIA ASSOCIADA OU NÃO A FÍBULA (I...,2025-01-01 00:27:21,10,AMIL,09178875,PARAFUSO DE BLOQUEIO 5.0MMX34MM - REF. 04005524S,5,NaN,NaN,NaN,10.0,10.0
3,791955,6,Contorno,BRUNO FARES DIAS,ORTOPEDIA E TRAUMATOLOGIA,RENATO BRASILEIRO DE LIMA,"{""solicitations"":[{""description"":""HASTE SUPRA ...",FRATURAS DE TÍBIA ASSOCIADA OU NÃO A FÍBULA (I...,2025-01-01 00:27:21,10,AMIL,09115231,PLACA TERÇO TUB LCP 3.5 7F C88 AÇO - REF: 241371,1,NaN,NaN,NaN,2.0,2.0
4,791955,6,Contorno,BRUNO FARES DIAS,ORTOPEDIA E TRAUMATOLOGIA,RENATO BRASILEIRO DE LIMA,"{""solicitations"":[{""description"":""HASTE SUPRA ...",FRATURAS DE TÍBIA ASSOCIADA OU NÃO A FÍBULA (I...,2025-01-01 00:27:21,10,AMIL,09111831,PARAFUSO CORTICAL AUTO ROSQUEANTE 3.5X14MM ACO...,6,NaN,NaN,NaN,12.0,12.0


In [10]:
# Célula 6: Criando a Tabela Agregada de Autorizações por Item/Hospital/Convênio

print("Iniciando a criação da tabela agregada de autorizações...")

if 'df_final' in locals() and not df_final.empty:
    # Colunas que usaremos para a análise de autorização
    auth_cols = ['mmd_qt_solicitada', 'mmd_qt_autorizada', 'mv_qt_solicitada', 'mv_qt_autorizada']
    
    # 1. Filtra o df_final para manter apenas linhas que têm alguma informação de autorização
    #    O 'how='all'' garante que a linha seja mantida se pelo menos um valor não for nulo.
    df_com_autorizacao = df_final.dropna(subset=auth_cols, how='all').copy()
    
    # 2. Define as chaves pelas quais vamos agrupar os dados
    grouping_keys = ['friendly_name', 'health_insurance_name', 'cd_pro_fat', 'ds_pro_fat']
    
    # 3. Define as agregações que queremos calcular (soma das quantidades)
    #    A lógica min_count=1 garante que a soma de nulos resulte em nulo, não em 0.
    aggregations = {
        'mmd_qt_solicitada': ('mmd_qt_solicitada', lambda x: x.sum(min_count=1)),
        'mmd_qt_autorizada': ('mmd_qt_autorizada', lambda x: x.sum(min_count=1)),
        'mv_qt_solicitada': ('mv_qt_solicitada', lambda x: x.sum(min_count=1)),
        'mv_qt_autorizada': ('mv_qt_autorizada', lambda x: x.sum(min_count=1))
    }

    # 4. Realiza o agrupamento e a agregação
    df_autorizacao_agg = df_com_autorizacao.groupby(grouping_keys).agg(**aggregations).reset_index()

    # (Bônus) Calcular a taxa de autorização para cada sistema pode ser útil
    df_autorizacao_agg['taxa_autorizacao_mmd'] = df_autorizacao_agg['mmd_qt_autorizada'] / df_autorizacao_agg['mmd_qt_solicitada']
    df_autorizacao_agg['taxa_autorizacao_mv'] = df_autorizacao_agg['mv_qt_autorizada'] / df_autorizacao_agg['mv_qt_solicitada']

    print(f"Tabela agregada de autorizações criada com sucesso. {len(df_autorizacao_agg)} registros únicos encontrados.")
    display(df_autorizacao_agg.head())

else:
    print("⚠️ DataFrame 'df_final' não encontrado ou vazio. A tabela agregada não pode ser criada.")
    df_autorizacao_agg = pd.DataFrame() # Cria um dataframe vazio para não quebrar a célula seguinte

Iniciando a criação da tabela agregada de autorizações...
Tabela agregada de autorizações criada com sucesso. 17733 registros únicos encontrados.


,friendly_name,health_insurance_name,cd_pro_fat,ds_pro_fat,mmd_qt_solicitada,mmd_qt_autorizada,mv_qt_solicitada,mv_qt_autorizada,taxa_autorizacao_mmd,taxa_autorizacao_mv
0,Betim-Contagem,AMIL,09002775,TROCARTE DESCARTAVEL 11MM REF- D11LT,1.0,1.0,2.0,2.0,1.0,1.0
1,Betim-Contagem,AMIL,09002776,TROCARTE ENDOPATH XCEL 12MM REF- D12LT,1.0,1.0,1.0,1.0,1.0,1.0
2,Betim-Contagem,AMIL,09002799,CARGA AZUL 75MM PARA GRAMPEADOR - REF- TCR75,NaN,NaN,2.0,2.0,NaN,1.0
3,Betim-Contagem,AMIL,09004603,AGULHA VERES 120MM DESCARTAVEL REF- UV120,2.0,2.0,2.0,2.0,1.0,1.0
4,Betim-Contagem,AMIL,09004626,GRAMPEADOR LINEAR CORTANTE 75MM REF- TLC75,NaN,NaN,1.0,1.0,NaN,1.0


In [9]:
# Célula 7 (ATUALIZADA): Salvar os dois relatórios em um único arquivo Excel

# Importa a biblioteca para manipulação de datas
from datetime import datetime

# Verifica se o DataFrame principal 'df_final' foi criado
if 'df_final' in locals() and not df_final.empty:
    try:
        # Define um nome de arquivo dinâmico com a data atual
        data_hoje = datetime.now().strftime('%Y-%m-%d')
        nome_arquivo = f'relatorio_materiais_todos_{data_hoje}.xlsx'

        # Usamos o pd.ExcelWriter para salvar em múltiplas abas
        print(f"Salvando relatórios no arquivo '{nome_arquivo}'...")
        with pd.ExcelWriter(nome_arquivo, engine='xlsxwriter') as writer:
            
            # Salva a base de dados transacional completa na primeira aba
            df_final.to_excel(writer, sheet_name='Base_Transacional_Completa', index=False)
            print("-> Aba 'Base_Transacional_Completa' salva.")
            
            # Salva a nova tabela agregada de autorizações na segunda aba
            if 'df_autorizacao_agg' in locals() and not df_autorizacao_agg.empty:
                df_autorizacao_agg.to_excel(writer, sheet_name='Resumo_Autorizacoes_Item', index=False)
                print("-> Aba 'Resumo_Autorizacoes_Item' salva.")
        
        print(f"\n✅ Relatório com 2 abas salvo com sucesso!")
        print("O arquivo está na mesma pasta onde este notebook Jupyter está localizado.")

    except Exception as e:
        print(f"❌ Ocorreu um erro ao tentar salvar o arquivo: {e}")
else:
    print("⚠️ O DataFrame 'df_final' está vazio ou não foi criado. Nenhum arquivo foi salvo.")

Salvando relatórios no arquivo 'relatorio_materiais_todos_2025-07-30.xlsx'...
-> Aba 'Base_Transacional_Completa' salva.
-> Aba 'Resumo_Autorizacoes_Item' salva.

✅ Relatório com 2 abas salvo com sucesso!
O arquivo está na mesma pasta onde este notebook Jupyter está localizado.
